# NLP — Carnet 07 : un mot comme un vecteur (co-occurrence, PPMI, SVD)

**Navigation** : [<< Précédent — 06. Recherche d'information (TF-IDF, BM25)](06-Recherche-Information-TFIDF-BM25.ipynb) | [Version Python](07-Vecteurs-de-Mots-Coocc-PPMI-SVD.ipynb) | [Version Python](07-Vecteurs-de-Mots-Coocc-PPMI-SVD.ipynb) | [Index de la série](README.md) | **07 (ce notebook)** | [Suivant — 08. Vecteurs de mots (word2vec) >>](08-Word2Vec-CBOW-SkipGram.ipynb)

**Objectifs d’apprentissage** : construire une matrice de co-occurrence à fenêtre symétrique, pondérer ses comptes par le PPMI, comparer des cosinus bruts et corrigés, puis réduire la matrice centrée par ligne avec une vraie SVD MathNet.

**Prérequis** : boucles et collections C#, logarithme, produit scalaire et notions de matrice. Le notebook utilise MathNet.Numerics 5.0.0 pour la décomposition en valeurs singulières.

**Durée estimée** : 45 min.

Un mot est représenté par son profil de co-occurrences : ce qui apparaît près de lui dans un corpus. Le comptage brut confond association et fréquence. Nous comparerons d’abord les lignes de comptes, puis leur pondération par le **PPMI** et une réduction de dimension par SVD. Tout le corpus est construit dans le notebook ; aucune donnée ni modèle externe n’est requis.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;
using System.Text.RegularExpressions;
using System.Xml.Linq;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.LinearAlgebra.Double;
using Microsoft.DotNet.Interactive.Formatting;
#nullable enable

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($"Environnement : .NET {Environment.Version}; MathNet.Numerics {typeof(DenseMatrix).Assembly.GetName().Version}; culture numérique invariant.");

Installed Packages MathNet.Numerics, 5.0.0

Environnement : .NET 10.0.14; MathNet.Numerics 5.0.0.0; culture numérique invariant.


## 1. Un corpus construit pour isoler les associations

Les treize phrases couvrent animaux, programmation et nourriture. Le mot `le` revient souvent et sert de hub. La tokenisation conserve les lettres latines et l’apostrophe, en minuscules ; le vocabulaire est trié pour rendre l’index déterministe.

In [2]:
var phrases = new[] {
    "le chien aboie dans le jardin",
    "le chat miaule sur le sofa",
    "un chat est un felin tres agile",
    "le chien et le chat vivent ensemble",
    "le chien aboie quand le chat miaule",
    "le code contient un bug vicieux",
    "le bug masque un algorithme lent",
    "l algorithme trie tout le code",
    "le code compile sans aucun bug",
    "le fruit est sucre et juteux",
    "la poire est un fruit mure",
    "le sucre rend le fruit delicieux",
    "une poire mure est un bon fruit"
};
var tokenizer = new Regex("[a-zà-öø-ÿ']+", RegexOptions.Compiled | RegexOptions.IgnoreCase);
List<string> Tokenize(string text) => tokenizer.Matches(text.ToLowerInvariant()).Select(m => m.Value).ToList();
var docs = phrases.Select(Tokenize).ToList();
var vocab = docs.SelectMany(d => d).Distinct(StringComparer.Ordinal).OrderBy(w => w, StringComparer.Ordinal).ToList();
var idx = vocab.Select((word, i) => (word, i)).ToDictionary(x => x.word, x => x.i, StringComparer.Ordinal);
var freq = vocab.ToDictionary(w => w, w => docs.Sum(d => d.Count(t => t == w)), StringComparer.Ordinal);
Console.WriteLine($"Corpus : {docs.Count} phrases, vocabulaire V={vocab.Count}. Six fréquences les plus élevées : {string.Join(", ", freq.OrderByDescending(p => p.Value).ThenBy(p => p.Key, StringComparer.Ordinal).Take(6).Select(p => $"{p.Key}={p.Value}"))}.");

Corpus : 13 phrases, vocabulaire V=41. Six fréquences les plus élevées : le=15, un=6, chat=4, est=4, fruit=4, bug=3.


## 2. Compter les voisins dans une fenêtre symétrique

Pour chaque position, on compte les positions distinctes situées à au plus trois mots, sans compter la position elle-même. La matrice est donc symétrique, et la répétition d’un même mot à deux positions différentes reste comptée.

In [3]:
const int W = 3;
var coo = new double[vocab.Count, vocab.Count];
foreach (var doc in docs)
{
    for (int i = 0; i < doc.Count; i++)
    {
        for (int j = Math.Max(0, i - W); j <= Math.Min(doc.Count - 1, i + W); j++)
        {
            if (i != j) coo[idx[doc[i]], idx[doc[j]]] += 1.0;
        }
    }
}
var row = new double[vocab.Count];
var col = new double[vocab.Count];
for (int i = 0; i < vocab.Count; i++)
for (int j = 0; j < vocab.Count; j++) { row[i] += coo[i, j]; col[j] += coo[i, j]; }
double N = row.Sum();
bool symmetric = Enumerable.Range(0, vocab.Count).All(i => Enumerable.Range(0, vocab.Count).All(j => coo[i,j] == coo[j,i]));
Console.WriteLine($"Fenêtre ±{W}, position centrale exclue : N={N:0}, matrice {vocab.Count}×{vocab.Count}, symétrique={symmetric}.");
Console.WriteLine($"Hubs reçus : le={col[idx["le"]]:0}, chien={col[idx["chien"]]:0}, code={col[idx["code"]]:0}, fruit={col[idx["fruit"]]:0}; coo(chien,le)={coo[idx["chien"],idx["le"]]:0}, coo(chien,chat)={coo[idx["chien"],idx["chat"]]:0}.");

Fenêtre ±3, position centrale exclue : N=336, matrice 41×41, symétrique=True.


Hubs reçus : le=55, chien=12, code=11, fruit=15; coo(chien,le)=6, coo(chien,chat)=1.


### Lecture du comptage

Les 13 phrases donnent 41 mots distincts et 336 événements. La colonne `le` reçoit 55 occurrences de voisinage : un cosinus calculé sur les comptes bruts peut donc refléter le profil partagé autour des mots-outils. Ce comptage est une mesure de proximité dans ce corpus, pas une étiquette sémantique.

## 3. Premier exercice — retrouver un compte

Écrivez `MonCooc(mot1, mot2)` pour lire le nombre de positions où `mot2` apparaît dans la fenêtre de `mot1`. La fixture contient un compte élevé, un compte différent de 1 et une paire absente. La garde exige ces trois valeurs : une constante ou un dictionnaire vide ne la satisfait pas.

In [4]:
// Exercice 1 — complétez le corps sans modifier la signature.
double? MonCooc(string mot1, string mot2)
{
    // TODO étudiant : renvoyer le compte coo(mot1, mot2), ou null si un mot est absent.
    Console.WriteLine("Exercice à compléter : co-occurrence");
    return null;
}
var coocFixture = new[] { ("chien", "le", 6.0), ("chien", "chat", 1.0), ("chien", "sofa", 0.0) };
bool CoocGuard(Func<string, string, double?> candidate) => coocFixture.All(t => candidate(t.Item1, t.Item2) is double value && Math.Abs(value - t.Item3) < 1e-12);
Console.WriteLine($"Garde de l’exercice 1 sur compte 6, compte 1 et zéro : résultat attendu après complétion = {CoocGuard(MonCooc)} (stub rejeté).");

Exercice à compléter : co-occurrence


Garde de l’exercice 1 sur compte 6, compte 1 et zéro : résultat attendu après complétion = False (stub rejeté).


## 4. Les comptes bruts et leur cosinus

Le cosinus compare les directions de deux lignes de la matrice. Une ligne nulle donne conventionnellement 0. Pour les voisins, on exclut le mot requête, on trie par score décroissant et les ex æquo par ordre lexical.

In [5]:
double Cosine(IReadOnlyList<double> a, IReadOnlyList<double> b)
{
    double dot = 0, na = 0, nb = 0;
    for (int i = 0; i < a.Count; i++) { dot += a[i] * b[i]; na += a[i] * a[i]; nb += b[i] * b[i]; }
    return na == 0 || nb == 0 ? 0 : dot / Math.Sqrt(na * nb);
}
List<double> RowVector(double[,] matrix, int i) => Enumerable.Range(0, matrix.GetLength(1)).Select(j => matrix[i,j]).ToList();
var rawVec = vocab.ToDictionary(w => w, w => RowVector(coo, idx[w]), StringComparer.Ordinal);
List<(string Word, double Score)> Nearest(IReadOnlyDictionary<string, List<double>> vectors, string word, int k = 3) =>
    vectors.Where(p => p.Key != word).Select(p => (Word: p.Key, Score: Cosine(vectors[word], p.Value)))
        .OrderByDescending(p => p.Score).ThenBy(p => p.Word, StringComparer.Ordinal).Take(k).ToList();
foreach (var word in new[] { "chien", "code", "fruit" })
    Console.WriteLine($"Voisins bruts de {word} : {string.Join("; ", Nearest(rawVec, word).Select(p => $"{p.Word} ({p.Score:F2})"))}");
Console.WriteLine($"Code : cos(code,chien)={Cosine(rawVec["code"],rawVec["chien"]):F3}; cos(code,bug)={Cosine(rawVec["code"],rawVec["bug"]):F3}.");

Voisins bruts de chien : aboie (0.83); quand (0.80); dans (0.80)


Voisins bruts de code : chien (0.66); compile (0.65); sans (0.65)


Voisins bruts de fruit : juteux (0.67); la (0.67); mure (0.67)


Code : cos(code,chien)=0.658; cos(code,bug)=0.605.


### Lecture des cosinus bruts

Dans ces lignes brutes, `cos(code, chien)` vaut environ 0,658, au-dessus de `cos(code, bug)` à 0,605. Le mot d’un autre domaine paraît plus proche que le congénère attendu : le cosinus compare des profils de comptes, où les contextes fréquents pèsent encore.

## 5. Le PPMI compare le compte à son attente

On calcule $\mathrm{PMI}(w_1,w_2)=\log\frac{(C(w_1,w_2)+\varepsilon)N}{R(w_1)C(w_2)+\varepsilon}$ avec $\varepsilon=10^{-9}$, la convention du notebook Python. Le PPMI conserve $\max(0,\mathrm{PMI})$. L’epsilon rend aussi finie la valeur intermédiaire d’une paire absente ; son PPMI reste nul.

In [6]:
const double Epsilon = 1e-9;
var pmi = new double[vocab.Count, vocab.Count];
var ppmi = new double[vocab.Count, vocab.Count];
for (int i = 0; i < vocab.Count; i++)
for (int j = 0; j < vocab.Count; j++)
{
    pmi[i,j] = Math.Log((coo[i,j] + Epsilon) * N / (row[i] * col[j] + Epsilon));
    ppmi[i,j] = Math.Max(0.0, pmi[i,j]);
}
foreach (var context in new[] { "le", "chat", "aboie", "dans" })
    Console.WriteLine($"chien / {context,-7} : PMI={pmi[idx["chien"],idx[context]]:+0.000}; PPMI={ppmi[idx["chien"],idx[context]]:F3}");
Console.WriteLine($"Contrôle paire absente chien/sofa : PMI fini={double.IsFinite(pmi[idx["chien"],idx["sofa"]])}, PPMI={ppmi[idx["chien"],idx["sofa"]]:F3}; epsilon={Epsilon:G}.");

chien / le      : PMI=+1.117; PPMI=1.117


chien / chat    : PMI=+0.499; PPMI=0.499


chien / aboie   : PMI=+1.723; PPMI=1.723


chien / dans    : PMI=+1.723; PPMI=1.723


Contrôle paire absente chien/sofa : PMI fini=True, PPMI=0.000; epsilon=1E-09.


### Lecture du PPMI

Pour `chien`, le PMI de `le` est 1,117, celui de `chat` 0,499, et ceux de `aboie` et `dans` 1,723. Le mot-outil ne disparaît donc pas : le PPMI garde une association observée au-delà de l’attente marginale. Il réduit l’effet de fréquence globale sans effacer une collocation réelle.

## 6. Deuxième exercice — calculer une valeur PPMI

Écrivez `MonPpmi(mot1, mot2)` avec les marges `row`, `col`, le total `N` et l’epsilon affiché. La garde vérifie une paire à PMI positif, une paire dont le PMI est ramené à zéro et une paire absente ; elle refuse une réponse constante.

In [7]:
// Exercice 2 — complétez le corps sans modifier la signature.
double? MonPpmi(string mot1, string mot2)
{
    // TODO étudiant : appliquer max(0, PMI) avec les marges et Epsilon.
    Console.WriteLine("Exercice à compléter : PPMI");
    return null;
}
var ppmiFixture = new[] { ("chien", "aboie", ppmi[idx["chien"],idx["aboie"]]), ("chien", "jardin", ppmi[idx["chien"],idx["jardin"]]), ("chien", "sofa", 0.0) };
bool PpmiGuard(Func<string, string, double?> candidate) => ppmiFixture.All(t => candidate(t.Item1, t.Item2) is double value && double.IsFinite(value) && Math.Abs(value - t.Item3) < 1e-9) && ppmiFixture.Select(t => t.Item3).Distinct().Count() >= 2;
Console.WriteLine($"Garde de l’exercice 2 sur PMI positif, PPMI nul et paire absente : résultat attendu après complétion = {PpmiGuard(MonPpmi)} (stub rejeté).");

Exercice à compléter : PPMI


Garde de l’exercice 2 sur PMI positif, PPMI nul et paire absente : résultat attendu après complétion = False (stub rejeté).


## 7. Centrer puis factoriser avec MathNet

On soustrait à chaque ligne sa propre moyenne, puis on demande à **MathNet.Numerics** une vraie SVD. On garde $k=2$ colonnes de $U$ pondérées par les deux premières valeurs singulières. Le centrage par ligne est le choix pédagogique de ce carnet, pas une règle universelle pour les embeddings.

In [8]:
var centered = new double[vocab.Count, vocab.Count];
var rowMeans = new double[vocab.Count];
for (int i = 0; i < vocab.Count; i++)
{
    for (int j = 0; j < vocab.Count; j++) rowMeans[i] += ppmi[i,j];
    rowMeans[i] /= vocab.Count;
    for (int j = 0; j < vocab.Count; j++) centered[i,j] = ppmi[i,j] - rowMeans[i];
}
var matrix = DenseMatrix.OfArray(centered);
var svd = matrix.Svd(true);
const int k = 2;
var embeddings = new Dictionary<string, List<double>>(StringComparer.Ordinal);
for (int i = 0; i < vocab.Count; i++)
    embeddings[vocab[i]] = Enumerable.Range(0, k).Select(j => svd.U[i,j] * svd.S[j]).ToList();
var reconstructed = svd.U * svd.W * svd.VT;
double reconstructionError = (matrix - reconstructed).FrobeniusNorm();
double maxGramResidual = 0.0;
for (int i = 0; i < vocab.Count; i++)
for (int j = 0; j < vocab.Count; j++)
{
    double embeddingDot = 0.0, svdGram = 0.0;
    for (int q = 0; q < k; q++)
    {
        embeddingDot += embeddings[vocab[i]][q] * embeddings[vocab[j]][q];
        svdGram += svd.U[i,q] * svd.U[j,q] * svd.S[q] * svd.S[q];
    }
    maxGramResidual = Math.Max(maxGramResidual, Math.Abs(embeddingDot - svdGram));
}
double maxRowCenterResidual = 0.0;
for (int i = 0; i < vocab.Count; i++)
{
    double residual = 0.0;
    for (int j = 0; j < vocab.Count; j++) residual += centered[i,j];
    maxRowCenterResidual = Math.Max(maxRowCenterResidual, Math.Abs(residual));
}
Console.WriteLine($"MathNet Svd(true) : dimensions {vocab.Count}×{vocab.Count}, rang conservé k={k}; σ₁={svd.S[0]:F6}, σ₂={svd.S[1]:F6}.");
Console.WriteLine($"Contrôle invariants : ||M−U·W·VT||F={reconstructionError:E3}; résidu Gram des embeddings rang 2={maxGramResidual:E3}; sommes de lignes centrées max={maxRowCenterResidual:E3}.");
foreach (var w in new[] { "chien", "chat", "code", "bug", "fruit", "le" }) Console.WriteLine($"{w,-10} [{embeddings[w][0]:0.000}; {embeddings[w][1]:0.000}]");

MathNet Svd(true) : dimensions 41×41, rang conservé k=2; σ₁=9.417715, σ₂=8.098939.


Contrôle invariants : ||M−U·W·VT||F=7.205E-014; résidu Gram des embeddings rang 2=1.776E-015; sommes de lignes centrées max=3.719E-015.


chien      [0.628; -1.573]


chat       [0.995; -1.617]


code       [-2.858; 0.106]


bug        [-2.765; 0.641]


fruit      [1.369; 1.120]


le         [0.485; -1.749]


### Lecture de la factorisation

La SVD complète reconstruit la matrice centrée avec un résidu de Frobenius proche de la précision machine ; le produit de Gram des vecteurs rang 2 est contrôlé à partir des facteurs, et les deux valeurs singulières retenues sont environ 9,418 et 8,099. Les coordonnées peuvent changer de signe d’une implémentation à l’autre : on compare plutôt les valeurs singulières, la reconstruction et les cosinus, qui ne dépendent pas d’un choix arbitraire de signe.

## 8. Troisième exercice — classer des voisins

Écrivez `PlusProches(vecteurs, mot, k)` : exclure `mot`, trier par cosinus décroissant puis par mot croissant en cas d’égalité. La fixture couvre le mot requête, deux voisins à score égal et un score inférieur. La garde vérifie les trois entrées, leur ordre exact et le départage lexical.

In [9]:
#nullable enable
// Exercice 3 — complétez le corps sans modifier la signature.
List<(string Word, double Score)>? PlusProches(IReadOnlyDictionary<string, List<double>> vectors, string word, int count)
{
    // TODO étudiant : calculer, exclure le mot, puis trier score décroissant et mot croissant.
    Console.WriteLine("Exercice à compléter : voisins cosinus");
    return null;
}
var tieFixture = new Dictionary<string, List<double>>(StringComparer.Ordinal) {
    ["requete"] = new() { 1, 0 }, ["zeta"] = new() { 0, 1 }, ["alpha"] = new() { 0, 1 }, ["loin"] = new() { -1, 0 }
};
bool NeighborGuard(Func<IReadOnlyDictionary<string, List<double>>, string, int, List<(string Word, double Score)>?> candidate)
{
    var result = candidate(tieFixture, "requete", 3);
    return result is { Count: 3 } && result.Select(p => p.Word).SequenceEqual(new[] { "alpha", "zeta", "loin" })
        && result.All(p => p.Word != "requete" && double.IsFinite(p.Score))
        && result.Zip(result.Skip(1)).All(pair => pair.First.Score >= pair.Second.Score)
        && Math.Abs(result[0].Score - result[1].Score) < 1e-12;
}
Console.WriteLine($"Garde des voisins : exclusion du mot, trois résultats, ordre décroissant et ex æquo lexical = {NeighborGuard(PlusProches)} (stub rejeté).");

Exercice à compléter : voisins cosinus


Garde des voisins : exclusion du mot, trois résultats, ordre décroissant et ex æquo lexical = False (stub rejeté).


## 9. Comparer les séparations entre domaines

Le test utilise les trois triplets animaux, programmation et nourriture. Pour chacun, on compare le cosinus brut au cosinus des vecteurs PPMI-SVD ; ces nombres décrivent uniquement ce petit corpus construit.

In [10]:
var pairs = new[] { (Word: "chien", Mate: "chat", Other: "code"), (Word: "code", Mate: "bug", Other: "fruit"), (Word: "fruit", Mate: "poire", Other: "chien") };
Console.WriteLine($"{"mot",-8}{"congénère",-12}{"autre domaine",-16}{"brut même",12}{"brut autre",12}{"PPMI-SVD même",16}{"PPMI-SVD autre",16}");
foreach (var p in pairs)
    Console.WriteLine($"{p.Word,-8}{p.Mate,-12}{p.Other,-16}{Cosine(rawVec[p.Word],rawVec[p.Mate]),12:F3}{Cosine(rawVec[p.Word],rawVec[p.Other]),12:F3}{Cosine(embeddings[p.Word],embeddings[p.Mate]),16:F3}{Cosine(embeddings[p.Word],embeddings[p.Other]),16:F3}");

mot     congénère   autre domaine      brut même  brut autre   PPMI-SVD même  PPMI-SVD autre


chien   chat        code                   0.735       0.658           0.985          -0.405


code    bug         fruit                  0.605       0.373           0.982          -0.750


fruit   poire       chien                  0.611       0.377           0.946          -0.301


### Lecture de la comparaison

Le tableau met face à face un mot du même domaine et un mot d’un autre domaine sous les deux représentations. La mesure utile est leur écart : le top-3 seul peut varier dans un espace à deux dimensions, tandis que les triplets fixés rendent visible le contraste choisi. Le corpus étant petit et construit, ce résultat illustre la méthode sans garantir des regroupements sur du texte général.

## 10. Figure des 41 mots dans le plan SVD

La figure est un SVG autonome, à fond blanc, avec axes, graduations et légende. Les 41 étiquettes restent présentes ; onze mots choisis sont colorés selon leur domaine et les autres apparaissent en gris.

In [11]:
string Xml(string value) => System.Security.SecurityElement.Escape(value) ?? "";
string F(double value) => value.ToString("0.###", CultureInfo.InvariantCulture);
var domain = new Dictionary<string,string>(StringComparer.Ordinal);
foreach (var w in new[] { "chien", "chat", "felin", "aboie", "miaule" }) domain[w] = "animaux";
foreach (var w in new[] { "code", "bug", "algorithme" }) domain[w] = "programme";
foreach (var w in new[] { "fruit", "poire", "sucre" }) domain[w] = "nourriture";
var colors = new Dictionary<string,string> { ["animaux"]="#c0392b", ["programme"]="#2c7be5", ["nourriture"]="#1e8e3e", ["autre"]="#777777" };
const int width = 1200, height = 900, left = 90, right = 280, top = 70, bottom = 95;
double xmin = embeddings.Values.Min(v => v[0]), xmax = embeddings.Values.Max(v => v[0]);
double ymin = embeddings.Values.Min(v => v[1]), ymax = embeddings.Values.Max(v => v[1]);
double padx = (xmax-xmin)*0.12, pady=(ymax-ymin)*0.12; xmin-=padx; xmax+=padx; ymin-=pady; ymax+=pady;
double sx(double x) => left + (x-xmin)/(xmax-xmin)*(width-left-right);
double sy(double y) => height-bottom - (y-ymin)/(ymax-ymin)*(height-top-bottom);
var svg = new StringBuilder();
svg.Append($"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'><rect width='100%' height='100%' fill='white'/>");
svg.Append("<text x='600' y='34' text-anchor='middle' font-family='sans-serif' font-size='22'>Vecteurs de mots — PPMI + SVD (k=2)</text>");
for (int tick=0; tick<=5; tick++) { double x=xmin+(xmax-xmin)*tick/5, y=ymin+(ymax-ymin)*tick/5; svg.Append($"<line x1='{F(sx(x))}' y1='{top}' x2='{F(sx(x))}' y2='{height-bottom}' stroke='#e5e7eb'/><text x='{F(sx(x))}' y='{height-bottom+24}' text-anchor='middle' font-family='sans-serif' font-size='13'>{F(x)}</text><line x1='{left}' y1='{F(sy(y))}' x2='{width-right}' y2='{F(sy(y))}' stroke='#e5e7eb'/><text x='{left-12}' y='{F(sy(y)+4)}' text-anchor='end' font-family='sans-serif' font-size='13'>{F(y)}</text>"); }
svg.Append($"<line x1='{left}' y1='{height-bottom}' x2='{width-right}' y2='{height-bottom}' stroke='#333'/><line x1='{left}' y1='{top}' x2='{left}' y2='{height-bottom}' stroke='#333'/><text x='{(left+width-right)/2}' y='{height-25}' text-anchor='middle' font-family='sans-serif' font-size='15'>Dimension 1 — coordonnées SVD</text><text transform='translate(24 {(top+height-bottom)/2}) rotate(-90)' text-anchor='middle' font-family='sans-serif' font-size='15'>Dimension 2 — coordonnées SVD</text>");
var labelBoxes = new List<(double X, double Y, double Width, double Height)>();
foreach (var w in vocab)
{
    var v = embeddings[w];
    var group = domain.GetValueOrDefault(w, "autre");
    var color = colors[group];
    double px = sx(v[0]), py = sy(v[1]);
    double labelWidth = w.Length * 7.4 + 3, labelHeight = 16;
    double labelX = px + 7, labelY = py - 5;
    bool placed = false;
    for (int radius = 13; radius <= 190 && !placed; radius += 11)
    {
        for (int direction = 0; direction < 16 && !placed; direction++)
        {
            double angle = 2 * Math.PI * direction / 16;
            double candidateX = px + radius * Math.Cos(angle);
            double candidateY = py + radius * Math.Sin(angle);
            double boxX = Math.Clamp(candidateX, left + 2, width - right - labelWidth - 2);
            double boxY = Math.Clamp(candidateY - labelHeight, top + 2, height - bottom - labelHeight - 2);
            bool overlaps = labelBoxes.Any(box => boxX < box.X + box.Width + 2 && boxX + labelWidth + 2 > box.X && boxY < box.Y + box.Height + 2 && boxY + labelHeight + 2 > box.Y);
            if (!overlaps) { labelX = boxX; labelY = boxY + labelHeight; placed = true; }
        }
    }
    labelBoxes.Add((labelX, labelY - labelHeight, labelWidth, labelHeight));
    if (Math.Abs(labelX - px) + Math.Abs(labelY - py) > 17)
        svg.Append($"<line x1='{F(px)}' y1='{F(py)}' x2='{F(labelX)}' y2='{F(labelY-4)}' stroke='#9ca3af' stroke-width='0.7'/>");
    svg.Append($"<circle cx='{F(px)}' cy='{F(py)}' r='4' fill='{color}'/><text x='{F(labelX)}' y='{F(labelY)}' font-family='sans-serif' font-size='12' fill='#202124'>{Xml(w)}</text>");
}
int legendY=170; foreach (var g in new[] { "animaux", "programme", "nourriture", "autre" }) { svg.Append($"<circle cx='{width-right+35}' cy='{legendY}' r='6' fill='{colors[g]}'/><text x='{width-right+50}' y='{legendY+5}' font-family='sans-serif' font-size='14'>{Xml(g)}</text>"); legendY+=30; }
svg.Append("</svg>");
var svgText = svg.ToString();
XDocument.Parse(svgText);
Formatter.Register<SvgOutput>((value, writer) => writer.Write(value.Content), "text/html");
display(new SvgOutput(svgText));
Console.WriteLine($"Figure SVG valide XML : {vocab.Count} étiquettes, {domain.Count} mots colorés sur trois domaines, axes et légende; fond blanc intégré.");
record SvgOutput(string Content);

Vecteurs de mots — PPMI + SVD (k=2) -3.419 -2.472 -2.259 -1.423 -1.099 -0.375 0.061 0.673 1.221 1.721 2.381 2.769 Dimension 1 — coordonnées SVD Dimension 2 — coordonnées SVD aboie agile algorithme aucun bon bug chat chien code compile contient dans delicieux ensemble est et felin fruit jardin juteux l la le lent masque miaule mure poire quand rend sans sofa sucre sur tout tres trie un une vicieux vivent animaux programme nourriture autre

Figure SVG valide XML : 41 étiquettes, 11 mots colorés sur trois domaines, axes et légende; fond blanc intégré.


### Lecture de la figure

Chaque point est un mot projeté dans les deux premières directions singulières de la matrice PPMI centrée. Les couleurs signalent les onze mots explicitement associés à un domaine ; les autres étiquettes sont conservées en gris. Les distances visibles résument cette projection et ne constituent pas une preuve que tous les mots d’un domaine formeraient un amas stable hors de ce corpus.

## À retenir

- **Compter avec une fenêtre explicite** : 13 phrases, 41 mots et 336 événements pour une fenêtre symétrique de ±3, sans compter la position centrale.
- **Le compte brut mélange fréquence et association** : `cos(code, chien)` vaut 0,658 contre 0,605 pour `cos(code, bug)`.
- **Le PPMI pondère l’écart à l’attente** avec la convention $\varepsilon=10^{-9}$ conservée de la version Python. Une paire absente donne un PPMI nul ; une collocation positive comme `chien/aboie` garde un PMI de 1,723.
- **MathNet fait la SVD réelle** de la matrice centrée par ligne. Les deux valeurs singulières conservées sont 9,418 et 8,099 ; les validations portent sur la reconstruction, les valeurs singulières et les cosinus, pas sur le signe arbitraire des coordonnées.
- **Le corpus est une illustration contrôlée**, pas une garantie d’amas sémantiques pour du texte général. Le centrage par ligne est un choix propre à cette leçon.

## Vérifiez votre compréhension

**1. Pourquoi `cos(code, chien)` peut-il dépasser `cos(code, bug)` sur les comptes bruts ?**

<details><summary>Réponse</summary>

Le cosinus compare des profils de co-occurrences. Les mots fréquents et les contextes communs contribuent à cette ressemblance même si les domaines diffèrent.
</details>

**2. Le PPMI doit-il annuler toute co-occurrence avec le mot fréquent `le` ?**

<details><summary>Réponse</summary>

Non. Il conserve les paires dont le compte dépasse l’attente calculée à partir des marges. Une collocation réelle peut donc garder un PMI positif même si le mot est fréquent.
</details>

**3. Que vérifie la reconstruction SVD complète, et pourquoi ne compare-t-on pas les coordonnées au signe près ?**

<details><summary>Réponse</summary>

La reconstruction contrôle que les facteurs MathNet décrivent la matrice centrée. Le signe de chaque vecteur singulier est arbitraire : on compare des invariants comme les valeurs singulières et les cosinus.
</details>

**4. Quelle limite faut-il garder en tête en lisant la figure ?**

<details><summary>Réponse</summary>

Elle montre une projection de 41 mots issus de treize phrases construites. Elle illustre les calculs, mais n’établit pas une garantie générale de regroupement sémantique.
</details>

## Pour aller plus loin

- [08 — word2vec (CBOW, skip-gram)](08-Word2Vec-CBOW-SkipGram.ipynb) apprend des vecteurs par prédiction plutôt que par factorisation directe.
- [09 — fastText](09-FastText-Subword-Vectors.ipynb) construit des représentations qui exploitent les sous-mots.
- [06 — TF-IDF et BM25](06-Recherche-Information-TFIDF-BM25.ipynb) présente le geste de pondération par rareté qui motive la comparaison avec le PPMI.
- Références : Church & Hanks (1990), Deerwester et al. (1990), Levy & Goldberg (2014).
